# 05_Webcam_Test_v2.ipynb
Test webcam với các checkpoint V2 đã train bằng Data Augmentation. Giữ nguyên pipeline webcam của notebook 03; chỉ đổi thư mục checkpoint sang `checkpoints_v2/<AUGMENT_MODE>`.

**Cách chạy:** chạy cell 1–4, sau đó chạy một trong các cell 5–7. Nhấn `q` trong cửa sổ webcam để thoát. Mặc định dùng checkpoint từ chế độ `all`; nếu đã train chế độ khác, đổi `AUGMENT_MODE` ở cell 2 cho đúng.


In [1]:
#cell 1: import thư viện và kiểm tra CUDA
import cv2
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
import mediapipe as mp
from mediapipe.tasks import python as mp_tasks
from mediapipe.tasks.python import vision

assert torch.cuda.is_available(), "CUDA chưa bật trong môi trường train-gpu"
device = torch.device("cuda")
print("CUDA:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "| MediaPipe:", mp.__version__, "| OpenCV:", cv2.__version__)


CUDA: NVIDIA GeForce RTX 3050 Ti Laptop GPU
PyTorch: 2.6.0+cu124 | MediaPipe: 1.1.0 | OpenCV: 5.0.0


In [2]:
#cell 2: đường dẫn dữ liệu và checkpoint V2
BASE = Path(r"C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321")
TASK_FILE = BASE / "hand_landmarker.task"
MANIFEST = BASE / "manifest.csv"
AUGMENT_MODE = "all"  # phải khớp chế độ đã train: all, rotation, noise, mirror, ...
CKPT_DIR = BASE / "checkpoints_v2" / AUGMENT_MODE

assert TASK_FILE.exists(), f"Không tìm thấy: {TASK_FILE}"
assert MANIFEST.exists(), f"Không tìm thấy: {MANIFEST}"
assert CKPT_DIR.exists(), f"Không tìm thấy thư mục checkpoint V2: {CKPT_DIR}"
import pandas as pd
manifest = pd.read_csv(MANIFEST)
T_MAX = int(manifest["khung"].max())
print("T_MAX:", T_MAX)
print("Checkpoint V2:", CKPT_DIR)
print("Các file hiện có:", [p.name for p in CKPT_DIR.glob("*.pt")])


T_MAX: 355
Checkpoint V2: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\checkpoints_v2\all
Các file hiện có: ['GRU.pt', 'TCN.pt', 'Transformer.pt']


In [3]:
#cell 3: định nghĩa đúng kiến trúc TCN, GRU và Transformer từ 02_Train.ipynb
class Block(nn.Module):
    def __init__(self, c, d):
        super().__init__()
        self.pad = 2 * d
        self.conv = nn.Conv1d(c, c, 3, dilation=d)
        self.drop = nn.Dropout(0.1)
    def forward(self, x):
        return x + self.drop(F.relu(self.conv(F.pad(x, (self.pad, 0)))))

class TCN(nn.Module):
    def __init__(self, c=64):
        super().__init__()
        self.inp = nn.Linear(126, c)
        self.blocks = nn.Sequential(*[Block(c, d) for d in (1, 2, 4, 8, 16)])
        self.out = nn.Linear(c, 3)
    def forward(self, x):
        return self.out(self.blocks(self.inp(x).transpose(1, 2)).transpose(1, 2))

class GRU(nn.Module):
    def __init__(self, c=64):
        super().__init__()
        self.inp = nn.Linear(126, c)
        self.rnn = nn.GRU(c, c, batch_first=True)
        self.out = nn.Linear(c, 3)
    def forward(self, x):
        return self.out(self.rnn(F.relu(self.inp(x)))[0])

class Trans(nn.Module):
    def __init__(self, c=64, T=T_MAX):
        super().__init__()
        self.inp = nn.Linear(126, c)
        self.pos = nn.Parameter(torch.randn(1, T, c) * 0.02)
        self.enc = nn.TransformerEncoder(nn.TransformerEncoderLayer(c, 4, 128, 0.1, batch_first=True), 2)
        self.out = nn.Linear(c, 3)
    def forward(self, x):
        t = x.shape[1]
        mask = nn.Transformer.generate_square_subsequent_mask(t, device=x.device)
        return self.out(self.enc(self.inp(x) + self.pos[:, :t], mask=mask))

LABELS = ["Paper", "Rock", "Scissors"]
print("Đã định nghĩa 3 kiến trúc model V2.")


Đã định nghĩa 3 kiến trúc model V2.


In [4]:
#cell 4: chuẩn hóa landmark, nội suy gần 120 FPS, vẽ khung xương và chạy webcam
HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4), (0,5),(5,6),(6,7),(7,8),
    (5,9),(9,10),(10,11),(11,12), (9,13),(13,14),(14,15),(15,16),
    (13,17),(0,17),(17,18),(18,19),(19,20)
]

def normalized_position(points):
    lm = np.asarray(points, dtype=np.float32)
    lm = lm - lm[:1]
    scale = np.linalg.norm(lm[9])
    if not np.isfinite(scale) or scale < 1e-6:
        scale = 1.0
    return (lm / scale).reshape(63)

def make_feature(pos, prev_pos):
    vel = np.zeros(63, dtype=np.float32) if prev_pos is None else pos - prev_pos
    return np.nan_to_num(np.concatenate([pos, vel])).astype(np.float32)

def draw_hand(frame, landmarks):
    h, w = frame.shape[:2]
    pts = [(int(p.x * w), int(p.y * h)) for p in landmarks]
    for a, b in HAND_CONNECTIONS:
        cv2.line(frame, pts[a], pts[b], (70, 220, 70), 2, cv2.LINE_AA)
    for x, y in pts:
        cv2.circle(frame, (x, y), 3, (40, 120, 255), -1, cv2.LINE_AA)

def run_webcam(model_name, model_class):
    ckpt_path = CKPT_DIR / f"{model_name}.pt"
    assert ckpt_path.exists(), f"Không tìm thấy checkpoint: {ckpt_path}. Hãy kiểm tra AUGMENT_MODE ở cell 2."
    model = model_class().to(device).eval()
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["w"])
    print(f"Đã nạp V2-{model_name}: {ckpt_path}")

    options = vision.HandLandmarkerOptions(
        base_options=mp_tasks.BaseOptions(model_asset_path=str(TASK_FILE)),
        running_mode=vision.RunningMode.VIDEO, num_hands=1)
    detector = vision.HandLandmarker.create_from_options(options)
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        detector.close()
        raise RuntimeError("Không mở được webcam. Hãy kiểm tra camera hoặc đổi VideoCapture(0).")

    history, prev_pos = [], None
    frame_id, last_time, fps = 0, time.perf_counter(), 0.0
    pred_text, conf_value = "Đang tìm bàn tay...", 0.0
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                print("Không đọc được khung hình webcam.")
                break
            frame = cv2.flip(frame, 1)
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            result = detector.detect_for_video(image, frame_id * 33)
            frame_id += 1

            if result.hand_landmarks:
                points = result.hand_landmarks[0]
                draw_hand(frame, points)
                pos = normalized_position([[p.x, p.y, p.z] for p in points])
                if prev_pos is None:
                    history.append(make_feature(pos, None))
                else:
                    prev_interp = prev_pos
                    for j in range(1, 5):
                        p = prev_pos + (pos - prev_pos) * (j / 4)
                        history.append(make_feature(p, prev_interp))
                        prev_interp = p
                prev_pos = pos
                history = history[-T_MAX:]
                x = torch.from_numpy(np.asarray(history, dtype=np.float32)).unsqueeze(0).to(device)
                with torch.inference_mode():
                    probs = torch.softmax(model(x)[0, -1], dim=-1)
                    conf, pred = probs.max(dim=-1)
                pred_text, conf_value = LABELS[int(pred.item())], float(conf.item())
            else:
                history.extend([np.zeros(126, dtype=np.float32)] * 4)
                history = history[-T_MAX:]
                prev_pos = None
                pred_text, conf_value = "Khong thay tay", 0.0

            now = time.perf_counter()
            dt = now - last_time
            if dt > 0:
                fps = 0.9 * fps + 0.1 / dt if fps else 1.0 / dt
            last_time = now
            cv2.putText(frame, f"Model: V2-{model_name}", (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 220, 255), 2, cv2.LINE_AA)
            cv2.putText(frame, f"Predict: {pred_text}", (15, 62), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2, cv2.LINE_AA)
            cv2.putText(frame, f"Confidence: {conf_value:.2f} (threshold 0.50)", (15, 94), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2, cv2.LINE_AA)
            cv2.putText(frame, f"FPS: {fps:.1f}", (15, 125), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 220, 255), 2, cv2.LINE_AA)
            cv2.imshow(f"Webcam - V2-{model_name} (press q to quit)", frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
    finally:
        cap.release()
        detector.close()
        cv2.destroyAllWindows()
        print(f"Đã dừng webcam V2-{model_name}.")


In [12]:
#cell 5: chạy thử V2-TCN
run_webcam("TCN", TCN)


Đã nạp V2-TCN: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\checkpoints_v2\all\TCN.pt
Đã dừng webcam V2-TCN.


In [11]:
#cell 6: chạy thử V2-GRU
run_webcam("GRU", GRU)


Đã nạp V2-GRU: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\checkpoints_v2\all\GRU.pt
Đã dừng webcam V2-GRU.


In [8]:
#cell 7: chạy thử V2-Transformer
run_webcam("Transformer", Trans)


Đã nạp V2-Transformer: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\checkpoints_v2\all\Transformer.pt
Đã dừng webcam V2-Transformer.


## Ghi chú
- Notebook chỉ nạp checkpoint V2, không train lại.
- Mặc định `AUGMENT_MODE = "all"`; cần trùng với thư mục checkpoint đã tạo trong `04_Trainv2.ipynb`.
- MediaPipe chạy trên CPU; model dự đoán chạy trên CUDA.
